In [6]:
library("R.matlab")
library("tidyverse")
library("afex")
library("BayesFactor")
library('emmeans')

In [7]:
extract_metrics <- function(filepath, group) {
  mat <- readMat(filepath)
  data.frame(
    Subject = basename(filepath),
    Group = group,
    Block = c('baseline', 'early_learning', 'late_learning'),
    meanRT = as.numeric(mat$meanRT[1:3]),
    meanMT = as.numeric(mat$meanMT[1:3])
  )
}

adult_files <- list.files('adult_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)
child_files <- list.files('children_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)

data_adult <- map_dfr(adult_files, ~extract_metrics(.x, 'adult'))
data_child <- map_dfr(child_files, ~extract_metrics(.x, 'child'))

data_all <- bind_rows(data_adult, data_child)

head(data_all)

,Subject,Group,Block,meanRT,meanMT
,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,VML_MEG_011_Final_Results.mat,adult,baseline,0.3590,1.018000
2,VML_MEG_011_Final_Results.mat,adult,early_learning,0.3308,1.136467
3,VML_MEG_011_Final_Results.mat,adult,late_learning,0.3260,1.093533
4,VML_MEG_012_2_Final_Results.mat,adult,baseline,0.3590,1.018000
5,VML_MEG_012_2_Final_Results.mat,adult,early_learning,0.3308,1.136467
6,VML_MEG_012_2_Final_Results.mat,adult,late_learning,0.3260,1.093533


In [8]:
# response time anova
anova_rt <- aov_ez(
  id = "Subject",
  dv = "meanRT",
  data = data_all,
  between = "Group",
  within = "Block"
)

print(anova_rt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group



Anova Table (Type 3 tests)

Response: meanRT
       Effect          df  MSE       F   ges p.value
1       Group       1, 22 0.03 8.95 **  .270    .007
2       Block 1.62, 35.58 0.00    0.73  .003    .461
3 Group:Block 1.62, 35.58 0.00    0.03 <.001    .941
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [9]:
# movement time anova
anova_mt <- aov_ez(
  id = "Subject",
  dv = "meanMT",
  data = data_all,
  between = "Group",
  within = "Block",
)

print(anova_mt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group



Anova Table (Type 3 tests)

Response: meanMT
       Effect          df  MSE         F  ges p.value
1       Group       1, 22 0.02      1.05 .033    .316
2       Block 1.33, 29.34 0.01 24.08 *** .239   <.001
3 Group:Block 1.33, 29.34 0.01      2.34 .030    .129
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [10]:
#response time bayes factor
data_all$Subject <- as.factor(data_all$Subject)
data_all$Group <- as.factor(data_all$Group)
data_all$Block <- as.factor(data_all$Block)

bf_rt <- anovaBF(
  meanRT ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)

print(bf_rt)

Bayes factor analysis
--------------
[1] Group + Subject                       : 4.93629   ±5.85%
[2] Block + Subject                       : 0.2153966 ±0.91%
[3] Group + Block + Subject               : 1.017754  ±2.4%
[4] Group + Block + Group:Block + Subject : 0.2075101 ±3.96%

Against denominator:
  meanRT ~ Subject 
---
Bayes factor type: BFlinearModel, JZS



In [11]:
#movement time bayes factor
bf_mt <- anovaBF(
  meanMT ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)

print(bf_mt)

Bayes factor analysis
--------------
[1] Group + Subject                       : 0.5091762 ±0.87%
[2] Block + Subject                       : 159804    ±0.65%
[3] Group + Block + Subject               : 98237.43  ±1.15%
[4] Group + Block + Group:Block + Subject : 87389.17  ±1.84%

Against denominator:
  meanMT ~ Subject 
---
Bayes factor type: BFlinearModel, JZS



In [12]:
print(anova_mt)
print(anova_rt)


Anova Table (Type 3 tests)

Response: meanMT
       Effect          df  MSE         F  ges p.value
1       Group       1, 22 0.02      1.05 .033    .316
2       Block 1.33, 29.34 0.01 24.08 *** .239   <.001
3 Group:Block 1.33, 29.34 0.01      2.34 .030    .129
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 
Anova Table (Type 3 tests)

Response: meanRT
       Effect          df  MSE       F   ges p.value
1       Group       1, 22 0.03 8.95 **  .270    .007
2       Block 1.62, 35.58 0.00    0.73  .003    .461
3 Group:Block 1.62, 35.58 0.00    0.03 <.001    .941
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [13]:
anova(lm(meanMT ~ Group * Block, data = data_all))

,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Group,1,0.02108027,0.021080275,2.252095,1.382027e-01
Block,2,0.20149263,0.100746315,10.763155,9.002753e-05
Group:Block,2,0.01886514,0.009432571,1.007722,3.705965e-01
Residuals,66,0.61777951,0.009360296,NA,NA


In [14]:
anova(lm(meanRT ~ Group * Block, data = data_all))

,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Group,1,2.253172e-01,2.253172e-01,24.451176802,5.521157e-06
Block,2,1.868913e-03,9.344563e-04,0.101406189,9.037064e-01
Group:Block,2,8.593606e-05,4.296803e-05,0.004662844,9.953483e-01
Residuals,66,6.081889e-01,9.214983e-03,NA,NA


In [15]:
# movement time pairwise comparisons
mt_model <- aov(meanMT ~ Group * Block + Error(Subject/Block), data = data_all)
mt_emm <- emmeans(mt_model, ~ Group * Block)
pairwise_mt <- pairs(mt_emm)
print(pairwise_mt)

Note: re-fitting model with sum-to-zero contrasts



 contrast                                    estimate     SE df t.ratio p.value
 adult baseline - child baseline              0.01684 0.0396 40   0.425  0.9981
 adult baseline - adult early_learning       -0.15445 0.0249 44  -6.206  <.0001
 adult baseline - child early_learning       -0.07457 0.0396 40  -1.881  0.4280
 adult baseline - adult late_learning        -0.08569 0.0249 44  -3.443  0.0151
 adult baseline - child late_learning        -0.07939 0.0396 40  -2.003  0.3587
 child baseline - adult early_learning       -0.17129 0.0396 40  -4.322  0.0013
 child baseline - child early_learning       -0.09141 0.0271 44  -3.379  0.0180
 child baseline - adult late_learning        -0.10253 0.0396 40  -2.587  0.1242
 child baseline - child late_learning        -0.09623 0.0271 44  -3.557  0.0110
 adult early_learning - child early_learning  0.07988 0.0396 40   2.015  0.3520
 adult early_learning - adult late_learning   0.06876 0.0249 44   2.763  0.0830
 adult early_learning - child late_learn

In [30]:
# response time pairwise comparisons
# pairwise comparisons between groups for meanRT across all blocks
rt_model <- aov(meanRT ~ Group * Block, data = data_all)
rt_emm <- emmeans(rt_model, ~ Group | Block)
pairwise_rt <- pairs(rt_emm, by = "Block")
print('response time pairwise')
print(pairwise_rt)

[1] "response time pairwise"
Block = baseline:
 contrast      estimate     SE df t.ratio p.value
 adult - child   -0.110 0.0393 66  -2.797  0.0068

Block = early_learning:
 contrast      estimate     SE df t.ratio p.value
 adult - child   -0.115 0.0393 66  -2.930  0.0046

Block = late_learning:
 contrast      estimate     SE df t.ratio p.value
 adult - child   -0.112 0.0393 66  -2.838  0.0060

Block = baseline:
 contrast      estimate     SE df t.ratio p.value
 adult - child   -0.110 0.0393 66  -2.797  0.0068

Block = early_learning:
 contrast      estimate     SE df t.ratio p.value
 adult - child   -0.115 0.0393 66  -2.930  0.0046

Block = late_learning:
 contrast      estimate     SE df t.ratio p.value
 adult - child   -0.112 0.0393 66  -2.838  0.0060



In [ ]:
# pairwise comparisons between blocks for meanMT within each group
mt_model <- aov(meanMT ~ Group * Block, data = data_all)
mt_emm <- emmeans(mt_model, ~ Block | Group)
pairwise_mt <- pairs(mt_emm, by = "Group")
print(pairwise_mt)

Group = adult:
 contrast                       estimate     SE df t.ratio p.value
 baseline - early_learning      -0.15445 0.0379 66  -4.070  0.0004
 baseline - late_learning       -0.08569 0.0379 66  -2.258  0.0690
 early_learning - late_learning  0.06876 0.0379 66   1.812  0.1736

Group = child:
 contrast                       estimate     SE df t.ratio p.value
 baseline - early_learning      -0.09141 0.0413 66  -2.216  0.0759
 baseline - late_learning       -0.09623 0.0413 66  -2.333  0.0582
 early_learning - late_learning -0.00482 0.0413 66  -0.117  0.9925

P value adjustment: tukey method for comparing a family of 3 estimates 


In [29]:
# pairwise comparisons between blocks for meanMT within each group
mt_model <- aov(meanMT ~ Group * Block, data = data_all)
mt_emm <- emmeans(mt_model, ~ Block | Group)
pairwise_mt <- pairs(mt_emm, by = "Block")
print("movement time pairwise")
print(pairwise_mt)

[1] "movement time pairwise"
Block = baseline:
 contrast      estimate     SE df t.ratio p.value
 adult - child   0.0168 0.0396 66   0.425  0.6723

Block = early_learning:
 contrast      estimate     SE df t.ratio p.value
 adult - child   0.0799 0.0396 66   2.015  0.0479

Block = late_learning:
 contrast      estimate     SE df t.ratio p.value
 adult - child   0.0063 0.0396 66   0.159  0.8741

Block = baseline:
 contrast      estimate     SE df t.ratio p.value
 adult - child   0.0168 0.0396 66   0.425  0.6723

Block = early_learning:
 contrast      estimate     SE df t.ratio p.value
 adult - child   0.0799 0.0396 66   2.015  0.0479

Block = late_learning:
 contrast      estimate     SE df t.ratio p.value
 adult - child   0.0063 0.0396 66   0.159  0.8741



In [25]:
anova_block_rt <- aov(meanRT ~ Block, data = data_all)
summary(anova_block_rt)

            Df Sum Sq  Mean Sq F value Pr(>F)
Block        2 0.0019 0.000934   0.077  0.926
Residuals   69 0.8336 0.012081               

In [ ]:
# pairwise comparisons between blocks for meanMT
mt_model_block <- aov(meanMT ~ Block, data = data_all)
mt_emm_block <- emmeans(mt_model_block, ~ Block)
pairwise_mt_block <- pairs(mt_emm_block)
print("movement time pairwise")
print(pairwise_mt_block)

[1] "movement time pairwise"
 contrast                       estimate     SE df t.ratio p.value
 baseline - early_learning       -0.1256 0.0282 69  -4.455  0.0001
 baseline - late_learning        -0.0905 0.0282 69  -3.212  0.0056
 early_learning - late_learning   0.0350 0.0282 69   1.243  0.4321

P value adjustment: tukey method for comparing a family of 3 estimates 
 contrast                       estimate     SE df t.ratio p.value
 baseline - early_learning       -0.1256 0.0282 69  -4.455  0.0001
 baseline - late_learning        -0.0905 0.0282 69  -3.212  0.0056
 early_learning - late_learning   0.0350 0.0282 69   1.243  0.4321

P value adjustment: tukey method for comparing a family of 3 estimates 
